# Comparaison des modèles

## 1. Import des librairies

In [1]:
import numpy as np
import pandas as pd
import arviz as az
import xarray as xr
import gc
import unicodedata
import re
from collections import Counter
import ast

## 2. Définition des fonctions utiles

In [2]:
def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def a_loglik(idata):
    """True si le groupe log_likelihood existe.
    ArviZ 1.x : idata.groups contient des chemins ('/posterior', '/log_likelihood'),
    donc  "log_likelihood" in idata.groups  est toujours False."""
    return any(g.strip("/") == "log_likelihood" for g in idata.groups)


def _g(res, *noms):
    """Lit un attribut du résultat az.loo selon la version d'ArviZ
    (1.x : elpd, p, elpd_i ; 0.x : elpd_loo, p_loo, loo_i)."""
    for n in noms:
        if hasattr(res, n):
            return getattr(res, n)
    raise AttributeError(noms)


def _extraire(idata, df_bayes, predicteurs_modele, col_voie):
    """Extrait du posterior tout ce qu'il faut pour recalculer eta_i."""
    post = idata.posterior
    n_chains, n_draws = post.sizes["chain"], post.sizes["draw"]
    seuils = post["threshold"].values.reshape(n_chains * n_draws, -1)   # (sims, K-1)
    betas = np.stack([post[p].values.reshape(-1)
                      for p in predicteurs_modele])                     # (P, sims)
    nom_u = f"1|{col_voie}"
    if nom_u in post:
        u = post[nom_u].values.reshape(n_chains * n_draws, -1)          # (sims, n_voies)
        # nom réel de la dimension des voies (ne pas supposer "<nom>_dim")
        dim_u = [d for d in post[nom_u].dims if d not in ("chain", "draw")][0]
        if dim_u in post[nom_u].coords:
            coords = np.asarray(post[nom_u].coords[dim_u].values).astype(str)
        else:                                   # repli : ordre des catégories de df_bayes
            coords = np.asarray(df_bayes[col_voie].cat.categories).astype(str)
        pos = {v: i for i, v in enumerate(coords)}
        route_codes = df_bayes[col_voie].astype(str).map(pos).values
        assert route_codes.min() >= 0, "Des voies de df_bayes sont absentes des offsets."
    else:
        u, route_codes = None, None        # modèle sans intercept aléatoire (A)
    X = df_bayes[predicteurs_modele].astype(float).values               # (n_obs, P)
    y = df_bayes["classe_cotation"].cat.codes.values                    # 0..K-1
    return post, n_chains, n_draws, seuils, betas, u, route_codes, X, y


def _calcule_plage(X, y, seuils, betas, u, route_codes, d0, d1, signe):
    """Log-likelihood des tirages [d0, d1), vectorisé. Rend (d1-d0, n_obs)."""
    n_obs, K = X.shape[0], seuils.shape[1] + 1
    eta = X @ betas[:, d0:d1]                       # (n_obs, chunk)
    if u is not None:
        eta = eta + u[d0:d1, route_codes].T         # (chunk, n_obs).T -> (n_obs, chunk)
    th = seuils[d0:d1]                              # (chunk, K-1)
    p_prev = np.zeros_like(eta)                     # P(Y <= -1) = 0
    ll = np.empty((d1 - d0, n_obs), dtype=np.float64)
    for k in range(K - 1):
        p_k = _sigmoid(signe * (eta - th[:, k][None, :]))     # P(Y <= k)
        logp = np.log(np.clip(p_k - p_prev, 1e-300, None))   # log P(Y = k)
        ll[:, y == k] = logp[y == k, :].T
        p_prev = p_k
    logp = np.log(np.clip(1.0 - p_prev, 1e-300, None))       # log P(Y = K-1)
    ll[:, y == K - 1] = logp[y == K - 1, :].T
    return ll


def valide_convention(idata, df_bayes, predicteurs_modele, col_voie="document_id",
                      n_sims=200):
    """
    Compare le calcul numpy au log-likelihood intégré sur les n_sims premiers
    tirages, pour les deux conventions de signe possibles :
      +1 : P(Y <= k) = sigmoid(eta - seuil_k)
      -1 : P(Y <= k) = sigmoid(seuil_k - eta)
    Rend (signe_validé, dict des écarts maximaux).
    """
    assert a_loglik(idata), \
        "Ce modèle n'a pas de log_likelihood stocké : rien à valider."
    (post, n_chains, n_draws, seuils, betas, u, route_codes, X, y) = _extraire(
        idata, df_bayes, predicteurs_modele, col_voie)
    var_nom = list(idata.log_likelihood.data_vars)[0]
    stocke = idata.log_likelihood[var_nom].values.reshape(n_chains * n_draws, -1)
    n_test = min(n_sims, stocke.shape[0])
    ecarts = {}
    for signe in (+1, -1):
        ll = _calcule_plage(X, y, seuils, betas, u, route_codes,
                           0, n_test, signe)
        ecarts[signe] = float(np.max(np.abs(ll - stocke[:n_test])))
    bon = min(ecarts, key=ecarts.get)
    if ecarts[bon] < 1e-4:
        print(f"Convention {bon:+d} VALIDÉE — écart max vs loglik intégré : "
              f"{ecarts[bon]:.2e}")
    else:
        print(f"⚠️ Aucune convention ne colle : écarts maximaux = {ecarts}")
    return bon, ecarts


def calcule_loglik(idata, df_bayes, predicteurs_modele, col_voie="document_id",
                   taille_chunk=100, signe=None, chemin_sortie=None):
    """
    Calcule et attache le groupe log_likelihood à idata.

    signe : convention validée par valide_convention (+1 par défaut si
    absente — lancez la validation sur C ou D d'abord !).
    chemin_sortie : si fourni, sauvegarde l'idata complété (NE PAS écraser
    l'original — garder le .nc brut intact).
    """
    (post, n_chains, n_draws, seuils, betas, u, route_codes, X, y) = _extraire(
        idata, df_bayes, predicteurs_modele, col_voie)
    n_obs, n_sims = X.shape[0], n_chains * n_draws
    if signe is None:
        if a_loglik(idata):
            signe, _ = valide_convention(idata, df_bayes, predicteurs_modele, col_voie)
        else:
            signe = +1
            print("⚠️ Pas de log_likelihood stocké : convention +1 par défaut. "
                  "Validez d'abord sur C ou D.")
    print(f"Calcul du log-likelihood : {n_sims} tirages × {n_obs} observations "
          f"(convention {signe:+d}, chunks de {taille_chunk})...")
    ll = np.empty((n_sims, n_obs), dtype=np.float32)   # float32 d'emblée : 2x moins de mémoire
    for d0 in range(0, n_sims, taille_chunk):
        d1 = min(d0 + taille_chunk, n_sims)
        ll[d0:d1] = _calcule_plage(X, y, seuils, betas, u, route_codes,
                                   d0, d1, signe)
    # Assemblage du groupe log_likelihood (float32 : moitié de la taille)
    da = xr.DataArray(
        ll.reshape(n_chains, n_draws, n_obs).astype(np.float32),
        dims=("chain", "draw", "obs"),
        coords={"chain": np.arange(n_chains), "draw": np.arange(n_draws)},
        name="classe_cotation",
    )
    if a_loglik(idata):
        del idata["log_likelihood"]
    idata["log_likelihood"] = xr.DataTree(dataset=xr.Dataset({"classe_cotation": da}))
    print(f"OK — log-likelihood ajouté (log moyen : {ll.mean():.2f}).")
    if chemin_sortie:
        idata.to_netcdf(chemin_sortie)
        print(f"idata complété sauvegardé : {chemin_sortie}")
    return idata

## Comparaison

In [3]:
# ══════════════════════════════════════════════════════════
# Cellule 2 : chargement et préparation de base (COMMUN)
# ══════════════════════════════════════════════════════════
ordre_cotations = [
    "1a", "1b", "1b+", "1c+",
    "2a", "2a+", "2b", "2b+", "2c", "2c+",
    "3a", "3a+", "3b", "3b+", "3c", "3c+",
    "4a", "4a+", "4b", "4b+", "4c", "4c+",
    "5a", "5a+", "5b", "5b+", "5c", "5c+",
    "6a", "6a+", "6b", "6b+", "6c", "6c+",
    "7a", "7a+", "7b", "7b+", "7c", "7c+",
    "8a", "8a+", "8b", "8b+", "8c", "8c+", "9a+",
]

df = pd.read_csv("annotation_finale_ponderee.csv")
df["difficulty"] = pd.Categorical(df["difficulty"], categories=ordre_cotations, ordered=True)

elements_frequents = ["dalle", "fissure", "dièdre", "traversée", "surplomb",
                      "dévers", "cheminée", "vire", "arête", "pilier",
                      "terrasse", "éperon"]
elements_intermediaires = ["écailles", "trou", "couloir", "cannelure", "réglette",
                           "goutte_eau", "crochet", "bac", "plat",
                           "petite_prise", "faille"]

# On garde TOUTES les colonnes (termes_ordonnes, pond_* seront
# nécessaires pour C et D) — la sélection se fait par liste de prédicteurs.
df_model = df.copy()
for e in elements_frequents:
    df_model[f"nb_{e}"] = pd.to_numeric(df_model[f"nb_{e}"]).clip(upper=3)
for e in elements_intermediaires:
    df_model[f"pres_{e}"] = (pd.to_numeric(df_model[f"nb_{e}"]) > 0).astype(int)
print(f"{len(df_model)} lignes chargées")

C:\Users\lunag\AppData\Local\Temp\ipykernel_34828\3033748808.py:15: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("annotation_finale_ponderee.csv")


KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════
# Cellule 3 : outils + fonction de préparation commune
# ══════════════════════════════════════════════════════════

def nettoyer_nom(nom):
    s = unicodedata.normalize("NFKD", nom).encode("ascii", "ignore").decode()
    return re.sub(r"\W+", "_", s).strip("_")     # ← "_" et pas "*" (bug corrigé)

ORDRE = [f"{n}{l}{p}" for n in range(1, 10) for l in "abc" for p in ["", "+"]]
RANG = {g: i for i, g in enumerate(ORDRE)}

def preparer_modele(df_model, predicteurs, col_voie="document_id",
                    cotation_min="4c+", cotation_max="7a+",
                    separer=False, seuil_fusion=400):
    """Reproduit exactement ta préparation : classes, (décomposition),
    fusion des rares, standardisation, réponse ordinale."""
    rang = df_model["difficulty"].astype("string").str.strip().str.lower() \
                .map(RANG).astype(float)
    bas, haut = RANG[cotation_min], RANG[cotation_max]
    data = pd.DataFrame({
        "classe_cotation": (rang.clip(lower=bas, upper=haut) - bas + 1),
        col_voie: df_model[col_voie],
    })
    noms_propres = {p: nettoyer_nom(p) for p in predicteurs}
    for p, propre in noms_propres.items():
        data[propre] = pd.to_numeric(df_model[p], errors="coerce")
    preds = list(noms_propres.values())
    data = data.dropna(subset=["classe_cotation"] + preds).copy()
    data["classe_cotation"] = data["classe_cotation"].astype(int)

    if separer:                                   # modèle A uniquement
        nouvelles = []
        for p in preds:
            moyenne = data.groupby(col_voie)[p].transform("mean")
            data[p + "_ecart"] = data[p] - moyenne
            data[p + "_moy_voie"] = moyenne
            nouvelles += [p + "_ecart", p + "_moy_voie"]
        preds = nouvelles

    effectifs = data["classe_cotation"].value_counts().sort_index()
    mapping = {c: (c - 1 if effectifs[c] < seuil_fusion and (c - 1) in effectifs.index
                   else c) for c in effectifs.index}
    data["classe_cotation"] = data["classe_cotation"].map(mapping)

    for p in preds:                               # standardisation + winsorisation
        x = data[p].astype(float)
        data[p] = ((x - x.mean()) / x.std()).clip(-10, 10)

    categories = sorted(data["classe_cotation"].unique())
    data["classe_cotation"] = pd.Categorical(data["classe_cotation"],
                                             categories=categories, ordered=True)
    data[col_voie] = data[col_voie].astype("category")
    print(f"  → {len(data)} lignes, {data[col_voie].nunique()} voies, "
          f"{data['classe_cotation'].nunique()} classes")
    return data, preds

In [ ]:
# ══════════════════════════════════════════════════════════
# Cellule 4 : les quatre préparations → PREPS
# ══════════════════════════════════════════════════════════
PREDICTEURS_B = ['nb_dalle', 'nb_fissure', 'nb_dièdre', 'nb_traversée',
                 'nb_surplomb', 'nb_dévers', 'nb_cheminée', 'nb_vire',
                 'nb_arête', 'nb_pilier', 'nb_terrasse', 'nb_éperon',
                 'pres_écailles', 'pres_trou', 'pres_couloir', 'pres_cannelure',
                 'pres_réglette', 'pres_goutte_eau', 'pres_crochet', 'pres_bac',
                 'pres_plat', 'pres_petite_prise', 'pres_faille',
                 'log_nb_terms_description']

PREPS = {}
print("Modèle B :");  PREPS["B"] = preparer_modele(df_model, PREDICTEURS_B)
print("Modèle A :");  PREPS["A"] = preparer_modele(df_model, PREDICTEURS_B, separer=True)

Modèle B :
  → 42861 lignes, 9299 voies, 14 classes
Modèle A :
  → 42861 lignes, 9299 voies, 14 classes


In [ ]:
# ── Cellule 4bis : modèle C — extraction des transitions ──

SEQ_COL = "termes_ordonnes"
SEUIL_TRANSITION = 400

def vers_liste(s):
    if isinstance(s, list):
        return s
    if pd.isna(s):
        return []
    try:
        v = ast.literal_eval(s)
        return v if isinstance(v, list) else [s]
    except (ValueError, SyntaxError):
        return [s]

seqs = df_model[SEQ_COL].apply(vers_liste)
freq = Counter()
for s in seqs:
    for a, b in zip(s, s[1:]):
        freq[(a, b)] += 1
retenues = sorted([t for t, n in freq.items() if n >= SEUIL_TRANSITION],
                  key=lambda t: -freq[t])
print(f"{len(retenues)} transitions retenues")     # ← doit afficher 24

noms_trans = []
for (a, b) in retenues:
    compte = Counter()
    for idx, s in zip(seqs.index, seqs):
        for x, y in zip(s, s[1:]):
            if (x, y) == (a, b):
                compte[idx] += 1
    nom = nettoyer_nom(f"trans_{a}_vers_{b}")
    df_model[nom] = pd.Series(compte).reindex(df_model.index,
                                              fill_value=0).astype(float)
    noms_trans.append(nom)

PREDICTEURS_C = PREDICTEURS_B + noms_trans
print("Modèle C :")
PREPS["C"] = preparer_modele(df_model, PREDICTEURS_C)

24 transitions retenues
Modèle C :
  → 42861 lignes, 9299 voies, 14 classes


In [ ]:
# ── Cellule 4ter : modèle D — pondérations ──
POND_RETENUES = ["dalle_ponderation", "fissure_ponderation", "dièdre_ponderation",
                 "traversée_ponderation", "surplomb_ponderation", "vire_ponderation",
                 "dévers_ponderation", "terrasse_ponderation"]
PREDICTEURS_D = PREDICTEURS_B + POND_RETENUES
print("Modèle D :")
PREPS["D"] = preparer_modele(df_model, PREDICTEURS_D)

Modèle D :
  → 42861 lignes, 9299 voies, 14 classes


In [ ]:
# ══════════════════════════════════════════════════════════
# Cellule 5 (remplace les anciennes 5 et 6) : log-likelihood numpy + LOO des 4 modèles
# ══════════════════════════════════════════════════════════
def signe_par_vraisemblance(idata, df_b, preds, n_tirages=50):
    """Choisit la convention de signe SANS log-likelihood stocké : la bonne est celle
    qui rend les données observées nettement plus vraisemblables sous la postérieure."""
    (post, nc, nd, seuils, betas, u, rc, X, y) = _extraire(idata, df_b, preds, "document_id")
    idx = np.linspace(0, nc * nd - 1, n_tirages).astype(int)      # tirages répartis sur les chaînes
    moyennes = {}
    for s in (+1, -1):
        moyennes[s] = float(np.mean([_calcule_plage(X, y, seuils, betas, u, rc, d, d + 1, s).sum()
                                     for d in idx]))
    bon = max(moyennes, key=moyennes.get)
    autre = -bon
    print(f"  log-vraisemblance moyenne : +1 → {moyennes[+1]:,.0f} | −1 → {moyennes[-1]:,.0f}"
          f"  ⇒ convention {bon:+d}")
    assert moyennes[bon] - moyennes[autre] > 100, "Écart trop faible : convention non identifiable."
    return bon


# Valeurs obtenues lors des premiers runs (avec log-likelihood de Bambi) : simple contrôle de cohérence
REFERENCE = {"A": -100209.2, "C": -86731.6, "D": -86510.4}

RESULTATS, SIGNE = {}, None
for nom in ["A", "B", "C", "D"]:
    print(f"── Modèle {nom} ──")
    df_b, preds = PREPS[nom]
    idata = az.from_netcdf(f"idata_{nom}.nc")
    if SIGNE is None:
        SIGNE = signe_par_vraisemblance(idata, df_b, preds)
    idata = calcule_loglik(idata, df_b, preds, signe=SIGNE)       # pas de sauvegarde (≈ 1 Go par modèle)
    loo = az.loo(idata, pointwise=True)
    # On ne garde que le nécessaire : le résultat complet contient les poids PSIS (≈ 2 Go par modèle)
    RESULTATS[nom] = {
        "elpd": float(_g(loo, "elpd", "elpd_loo")),
        "se": float(loo.se),
        "p": float(_g(loo, "p", "p_loo")),
        "elpd_i": np.asarray(_g(loo, "elpd_i", "loo_i").values).copy(),
        "pareto_k": np.asarray(loo.pareto_k.values).copy(),
    }
    msg = f"  {nom} → elpd_loo = {RESULTATS[nom]['elpd']:,.1f}"
    if nom in REFERENCE:
        msg += f"  (premier run : {REFERENCE[nom]:,.1f} ; écart {RESULTATS[nom]['elpd'] - REFERENCE[nom]:+.1f})"
    print(msg)
    del idata, loo
    gc.collect()
print("Convention commune aux quatre modèles :", SIGNE)


── Modèle A ──
  log-vraisemblance moyenne : +1 → -24,591,819 | −1 → -82,247  ⇒ convention -1
Calcul du log-likelihood : 6000 tirages × 42861 observations (convention -1, chunks de 100)...
OK — log-likelihood ajouté (log moyen : -1.92).


c:\Python313\Lib\site-packages\arviz_stats\loo\loo_helper.py:1169: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


  A → elpd_loo = -86,591.4  (premier run : -100,209.2 ; écart +13617.8)
── Modèle B ──
Calcul du log-likelihood : 6000 tirages × 42861 observations (convention -1, chunks de 100)...
OK — log-likelihood ajouté (log moyen : -1.92).


c:\Python313\Lib\site-packages\arviz_stats\loo\loo_helper.py:1169: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


  B → elpd_loo = -86,729.1
── Modèle C ──
Calcul du log-likelihood : 6000 tirages × 42861 observations (convention -1, chunks de 100)...
OK — log-likelihood ajouté (log moyen : -1.92).


c:\Python313\Lib\site-packages\arviz_stats\loo\loo_helper.py:1169: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


  C → elpd_loo = -86,729.8  (premier run : -86,731.6 ; écart +1.8)
── Modèle D ──
Calcul du log-likelihood : 6000 tirages × 42861 observations (convention -1, chunks de 100)...
OK — log-likelihood ajouté (log moyen : -1.91).


c:\Python313\Lib\site-packages\arviz_stats\loo\loo_helper.py:1169: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


  D → elpd_loo = -86,512.0  (premier run : -86,510.4 ; écart -1.6)
Convention commune aux quatre modèles : -1


In [ ]:
# ══════════════════════════════════════════════════════════
# Cellule 6 : les tableaux de comparaison
# ══════════════════════════════════════════════════════════
lignes = {n: {"elpd_loo": round(r["elpd"], 1),
              "se": round(r["se"], 1),
              "p_loo": round(r["p"], 1),
              "part_k>0.7": round(float((r["pareto_k"] > 0.7).mean()), 3)}
          for n, r in RESULTATS.items()}
print(pd.DataFrame(lignes).T.sort_values("elpd_loo", ascending=False))

loo_i = {n: r["elpd_i"] for n, r in RESULTATS.items()}
meilleur = max(loo_i, key=lambda k: loo_i[k].sum())
comp = []
for nom, li in loo_i.items():
    diff = loo_i[meilleur] - li
    comp.append({"modele": nom,
                 "elpd_loo": round(li.sum(), 1),
                 "diff_elpd": round(diff.sum(), 1),
                 "se_diff": round(float(np.sqrt(len(li) * diff.var(ddof=1))), 1)
                 if nom != meilleur else 0.0})
print(f"\nModèle de référence : {meilleur}")
print(pd.DataFrame(comp).sort_values("diff_elpd").to_string(index=False))


   elpd_loo     se   p_loo  part_k>0.7
D  -86512.0  179.4  7694.5       0.004
A  -86591.4  179.4  7531.1       0.002
B  -86729.1  179.0  7685.6       0.004
C  -86729.8  179.2  7714.5       0.004

Modèle de référence : D
modele  elpd_loo  diff_elpd  se_diff
     D  -86512.0        0.0      0.0
     A  -86591.4       79.4     32.7
     B  -86729.1      217.1     22.7
     C  -86729.8      217.8     23.7
